# Locked Retrieval Evaluation

This notebook runs the frozen retriever on the locked evaluation dataset.

The retrieval configuration and locked questions must not be changed based on these results. The first run will be preserved as the unbiased baseline.

In [ ]:
import hashlib
import json
from pathlib import Path


PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent


config_path = PROJECT_ROOT / "evaluation" / "retrieval_config.json"
questions_path = PROJECT_ROOT / "evaluation" / "locked_questions.json"
hash_path = PROJECT_ROOT / "evaluation" / "locked_questions.sha256"


config = json.loads(config_path.read_text(encoding="utf-8"))
locked_questions = json.loads(
    questions_path.read_text(encoding="utf-8")
)

stored_hash = hash_path.read_text(
    encoding="utf-8"
).split()[0]

actual_hash = hashlib.sha256(
    questions_path.read_bytes()
).hexdigest()


assert stored_hash == actual_hash
assert config["locked_evaluation_dataset"]["sha256"] == actual_hash
assert config["status"] == "frozen_before_locked_evaluation"


print("Project root:", PROJECT_ROOT)
print("Locked questions:", len(locked_questions))
print("Frozen hash:", actual_hash)
print("Configuration status:", config["status"])

## Load Evaluation Data

Retrieval metrics apply only to answerable questions because refusal questions intentionally have no gold pages.

Refusal behaviour will be evaluated later at the answer-generation stage.

In [ ]:
import pandas as pd


chunks_path = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "chunks.jsonl"
)

chunks = pd.read_json(
    chunks_path,
    lines=True,
)

questions = pd.DataFrame(locked_questions)

answerable_questions = questions.loc[
    questions["answerable"]
].copy()

refusal_questions = questions.loc[
    ~questions["answerable"]
].copy()
#.copy() create a separate copy of the DataFrame.

assert len(chunks) == 368
assert chunks["chunk_id"].is_unique
assert len(answerable_questions) == 23
assert len(refusal_questions) == 5


print("Chunks:", len(chunks))
print("Answerable questions:", len(answerable_questions))
print("Refusal questions:", len(refusal_questions))
print(
    answerable_questions["category"]
    .value_counts()
    .to_string()
)

## Build the Frozen Hybrid Retriever

The hybrid retriever combines:

1. TF-IDF lexical retrieval
2. BGE semantic retrieval
3. Weighted reciprocal rank fusion

All settings come from the frozen configuration. We must not change them after seeing locked results.

In [ ]:
from sentence_transformers import SentenceTransformer

from aviation_rag.retrieval import (
    HybridRetriever,
    SemanticRetriever,
    TfidfRetriever,
)


semantic_config = config["semantic_retriever"]
hybrid_config = config["hybrid_retriever"]


bge_model = SentenceTransformer(
    semantic_config["model"]
)

tfidf_retriever = TfidfRetriever(
    chunks=chunks
)

semantic_retriever = SemanticRetriever(
    chunks=chunks,
    model=bge_model,
    query_prefix=semantic_config["query_instruction"],
)

hybrid_retriever = HybridRetriever(
    lexical_retriever=tfidf_retriever,
    semantic_retriever=semantic_retriever,
    candidate_k=hybrid_config["candidate_k"],
    rrf_constant=hybrid_config["rrf_constant"],
    lexical_weight=hybrid_config["lexical_weight"],
)


print("Semantic model:", semantic_config["model"])
print(
    "Embedding matrix:",
    semantic_retriever.chunk_embeddings.shape,
)
print("Candidate K:", hybrid_retriever.candidate_k)
print("RRF constant:", hybrid_retriever.rrf_constant)
print("Lexical weight:", hybrid_retriever.lexical_weight)
print("Semantic weight:", hybrid_retriever.semantic_weight)

## Locked Evaluation Preflight

This check confirms that every answerable question has valid, retrieval-eligible gold pages and that no earlier locked result files will be overwritten.

In [ ]:
from aviation_rag.evaluation_data import validate_gold_evidence


pages_path = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "pages.jsonl"
)

results_path = (
    PROJECT_ROOT
    / "evaluation"
    / "locked_retrieval_metrics.csv"
)

rankings_path = (
    PROJECT_ROOT
    / "evaluation"
    / "locked_retrieval_rankings.jsonl"
)


pages = pd.read_json(
    pages_path,
    lines=True,
)

evidence_errors = validate_gold_evidence(
    questions=locked_questions,
    pages=pages,
)


assert evidence_errors == []
assert not results_path.exists()
assert not rankings_path.exists()
assert (
    config["locked_evaluation_dataset"]["retrieval_run_status"]
    == "not_run"
)


print("Gold-evidence errors:", len(evidence_errors))
print("Metrics output exists:", results_path.exists())
print("Rankings output exists:", rankings_path.exists())
print("Locked evaluation is ready.")

## First Locked Retrieval Run

This is the first unbiased evaluation of the frozen retriever.

The retriever returns five ranked chunks for each answerable question. Metrics are calculated against verified gold pages. Both summary metrics and detailed rankings are saved for failure analysis.

In [ ]:
evaluation_records = []
ranking_records = []

top_k_values = (1, 3, 5)
maximum_k = max(top_k_values)


assert not results_path.exists()
assert not rankings_path.exists()


for item in locked_questions:
    if not item["answerable"]:
        continue

    gold_pages = {
        (document_id, int(page_number))
        for document_id, page_numbers
        in item["gold_pages"].items()
        for page_number in page_numbers
    }

    retrieved = hybrid_retriever.retrieve(
        query=item["question"],
        top_k=maximum_k,
    )

    retrieved_pages = list(
        zip(
            retrieved["document_id"],
            retrieved["page_number"].astype(int),
        )
    )

    metric_record = {
        "question_id": item["question_id"],
        "category": item["category"],
        "gold_page_count": len(gold_pages),
    }

    for k in top_k_values:
        top_k_pages = set(retrieved_pages[:k])
        matched_pages = gold_pages & top_k_pages

        metric_record[f"recall_at_{k}"] = (
            len(matched_pages) / len(gold_pages)
        )

        metric_record[f"hit_at_{k}"] = int(
            bool(matched_pages)
        )

    evaluation_records.append(metric_record)

    for rank, row in enumerate(
        retrieved.itertuples(index=False),
        start=1,
    ):
        page_key = (
            row.document_id,
            int(row.page_number),
        )

        ranking_records.append(
            {
                "question_id": item["question_id"],
                "question": item["question"],
                "category": item["category"],
                "rank": rank,
                "chunk_id": row.chunk_id,
                "document_id": row.document_id,
                "page_number": int(row.page_number),
                "score": float(row.score),
                "is_gold_page": page_key in gold_pages,
            }
        )


locked_metrics = pd.DataFrame(evaluation_records)
locked_rankings = pd.DataFrame(ranking_records)


locked_metrics.to_csv(
    results_path,
    index=False,
)

locked_rankings.to_json(
    rankings_path,
    orient="records",
    lines=True,
    force_ascii=False,
)


metric_columns = [
    "recall_at_1",
    "recall_at_3",
    "recall_at_5",
    "hit_at_1",
    "hit_at_3",
    "hit_at_5",
]


print("Evaluated questions:", len(locked_metrics))
print("Saved ranking rows:", len(locked_rankings))
print("\nMean locked metrics:")
print(
    locked_metrics[metric_columns]
    .mean()
    .round(3)
)

## Preserve First-Run Results

The first locked result is saved before failure analysis. This prevents later experiments from replacing the unbiased baseline.

In [ ]:
from datetime import datetime, timezone


run_metadata_path = (
    PROJECT_ROOT
    / "evaluation"
    / "locked_retrieval_run.json"
)


assert not run_metadata_path.exists()


mean_metrics = {
    column: round(
        float(locked_metrics[column].mean()),
        6,
    )
    for column in metric_columns
}


run_metadata = {
    "run_type": "first_locked_retrieval_evaluation",
    "run_timestamp_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "locked_dataset_sha256": actual_hash,
    "answerable_questions": len(locked_metrics),
    "retrieved_chunks_per_question": maximum_k,
    "retriever": {
        "chunking": config["chunking"],
        "lexical_retriever": config["lexical_retriever"],
        "semantic_retriever": config["semantic_retriever"],
        "hybrid_retriever": config["hybrid_retriever"],
    },
    "mean_metrics": mean_metrics,
    "outputs": {
        "metrics": "evaluation/locked_retrieval_metrics.csv",
        "rankings": "evaluation/locked_retrieval_rankings.jsonl",
    },
}


run_metadata_path.write_bytes(
    (
        json.dumps(
            run_metadata,
            indent=2,
            ensure_ascii=False,
        )
        + "\n"
    ).encode("utf-8")
)


config["locked_evaluation_dataset"][
    "retrieval_run_status"
] = "completed"

config["locked_evaluation_dataset"][
    "first_run_metrics"
] = mean_metrics

config["locked_evaluation_dataset"][
    "first_run_metadata"
] = "evaluation/locked_retrieval_run.json"


config_path.write_bytes(
    (
        json.dumps(
            config,
            indent=2,
            ensure_ascii=False,
        )
        + "\n"
    ).encode("utf-8")
)


print("First locked run preserved.")
print("Metadata:", run_metadata_path)
print("Status:", config["locked_evaluation_dataset"]["retrieval_run_status"])

## Question-Level Failure Analysis

A retrieval failure is not one single problem. We separate:

1. Complete success: every gold page was retrieved
2. Partial success: at least one gold page was retrieved, but not all
3. Complete miss: no gold page was retrieved

In [ ]:
category_summary = (
    locked_metrics
    .groupby("category")[metric_columns]
    .mean()
    .round(3)
)

category_summary.insert(
    0,
    "question_count",
    locked_metrics["category"].value_counts(),
)


failure_summary = (
    locked_metrics.loc[
        locked_metrics["recall_at_5"] < 1
    ]
    .merge(
        questions[
            [
                "question_id",
                "question",
                "difficulty",
                "gold_pages",
            ]
        ],
        on="question_id",
        how="left",
    )
    .copy()
)

failure_summary["failure_type"] = (
    failure_summary["hit_at_5"]
    .map(
        {
            1: "partial_success",
            0: "complete_miss",
        }
    )
)


print("Category summary:")
print(category_summary.to_string())

print("\nQuestions without full Recall@5:")
print(
    failure_summary[
        [
            "question_id",
            "category",
            "difficulty",
            "gold_page_count",
            "recall_at_5",
            "hit_at_5",
            "failure_type",
        ]
    ].to_string(index=False)
)

## Inspect Synthesis Failures

We now inspect what the retriever returned instead of guessing why it failed. No retrieval settings are changed.

In [ ]:
synthesis_failures = failure_summary.loc[
    failure_summary["category"]
    == "cross_document_synthesis"
]


for question_id in synthesis_failures["question_id"]:
    item = next(
        question
        for question in locked_questions
        if question["question_id"] == question_id
    )

    retrieved_rows = locked_rankings.loc[
        locked_rankings["question_id"] == question_id,
        [
            "chunk_id",
            "rank",
            "document_id",
            "page_number",
            "score",
            "is_gold_page",
        ],
    ].copy()

    retrieved_rows = retrieved_rows.merge(
    chunks[
        [
            "chunk_id",
            "chunk_text",
        ]
    ],
    on="chunk_id",
    how="left",
    validate="many_to_one",
    )

    retrieved_rows["text_preview"] = (
        retrieved_rows["chunk_text"]
        .str.replace(r"\s+", " ", regex=True)
        .str.slice(0, 180)
    )

    print("=" * 80)
    print("Question ID:", question_id)
    print("Question:", item["question"])
    print("Gold pages:", item["gold_pages"])
    print("\nRetrieved top five:")

    print(
        retrieved_rows[
            [
                "rank",
                "document_id",
                "page_number",
                "score",
                "is_gold_page",
                "text_preview",
            ]
        ].to_string(index=False)
    )

## Missing Gold Pages

This table separates retrieved evidence from missing evidence. It helps determine whether failures come from wrong documents, missing amendment versions, or incomplete multi-page coverage.

In [ ]:
missing_page_records = []


for item in locked_questions:
    if not item["answerable"]:
        continue

    question_metrics = locked_metrics.loc[
        locked_metrics["question_id"]
        == item["question_id"]
    ].iloc[0]

    if question_metrics["recall_at_5"] == 1:
        continue

    gold_pages = {
        (document_id, int(page_number))
        for document_id, page_numbers
        in item["gold_pages"].items()
        for page_number in page_numbers
    }

    retrieved_pages = {
        (
            row.document_id,
            int(row.page_number),
        )
        for row in locked_rankings.loc[
            locked_rankings["question_id"]
            == item["question_id"]
        ].itertuples(index=False)
    }

    found_pages = sorted(
        gold_pages & retrieved_pages
    )

    missing_pages = sorted(
        gold_pages - retrieved_pages
    )

    missing_page_records.append(
        {
            "question_id": item["question_id"],
            "category": item["category"],
            "found_gold_pages": found_pages,
            "missing_gold_pages": missing_pages,
        }
    )


missing_pages_df = pd.DataFrame(
    missing_page_records
)

print(
    missing_pages_df.to_string(index=False)
)

## Component Rank Diagnosis

The hybrid retriever only fuses the top 20 candidates from TF-IDF and BGE. If a gold page ranks below 20 in both components, the hybrid system never gets a chance to rank it.

In [ ]:
component_rank_records = []

questions_by_id = {
    item["question_id"]: item
    for item in locked_questions
}


for failure in missing_page_records:
    item = questions_by_id[
        failure["question_id"]
    ]

    tfidf_results = tfidf_retriever.retrieve(
        query=item["question"],
        top_k=len(chunks),
    )

    semantic_results = semantic_retriever.retrieve(
        query=item["question"],
        top_k=len(chunks),
    )

    for document_id, page_number in failure[
        "missing_gold_pages"
    ]:
        tfidf_matches = tfidf_results.index[
            (
                tfidf_results["document_id"]
                == document_id
            )
            & (
                tfidf_results["page_number"]
                == page_number
            )
        ]

        semantic_matches = semantic_results.index[
            (
                semantic_results["document_id"]
                == document_id
            )
            & (
                semantic_results["page_number"]
                == page_number
            )
        ]

        tfidf_rank = (
            int(tfidf_matches[0]) + 1
            if len(tfidf_matches)
            else None
        )

        semantic_rank = (
            int(semantic_matches[0]) + 1
            if len(semantic_matches)
            else None
        )

        component_rank_records.append(
            {
                "question_id": item["question_id"],
                "document_id": document_id,
                "page_number": page_number,
                "tfidf_best_rank": tfidf_rank,
                "bge_best_rank": semantic_rank,
                "entered_hybrid_candidates": (
                    tfidf_rank <= 20
                    or semantic_rank <= 20
                ),
            }
        )


component_ranks = pd.DataFrame(
    component_rank_records
)

print(
    component_ranks.to_string(index=False)
)

In [ ]:
hybrid_rank_records = []


for failure in missing_page_records:
    item = questions_by_id[
        failure["question_id"]
    ]

    hybrid_candidates = hybrid_retriever.retrieve(
        query=item["question"],
        top_k=40,
    )

    for document_id, page_number in failure[
        "missing_gold_pages"
    ]:
        matches = hybrid_candidates.index[
            (
                hybrid_candidates["document_id"]
                == document_id
            )
            & (
                hybrid_candidates["page_number"]
                == page_number
            )
        ]

        hybrid_rank_records.append(
            {
                "question_id": item["question_id"],
                "document_id": document_id,
                "page_number": page_number,
                "hybrid_best_rank": (
                    int(matches[0]) + 1
                    if len(matches)
                    else None
                ),
            }
        )


hybrid_missing_ranks = pd.DataFrame(
    hybrid_rank_records
)

print(
    hybrid_missing_ranks.to_string(index=False)
)

### Interpretation

The temporal and amendment failures are mostly near misses. Their missing evidence pages ranked between 6 and 11, just outside the evaluated top five. This shows that the retriever often recognizes the correct evidence but does not rank every required page highly enough.

The synthesis failures are more severe. Some gold pages ranked between 9 and 29, while two pages did not enter the hybrid candidate pool. The full multi-intent questions caused evidence components to compete with each other.

Increasing Top K alone is not a sufficient solution. It may recover several near-miss pages, but it also adds irrelevant context and still does not recover all synthesis evidence. Query decomposition, metadata filtering, and controlled adjacent-page expansion should be considered for a later system version.

In [ ]:
failures_path = (
    PROJECT_ROOT
    / "evaluation"
    / "locked_retrieval_failures.csv"
)

missing_ranks_path = (
    PROJECT_ROOT
    / "evaluation"
    / "locked_missing_page_ranks.csv"
)


failure_summary.to_csv(
    failures_path,
    index=False,
)

(
    component_ranks
    .merge(
        hybrid_missing_ranks,
        on=[
            "question_id",
            "document_id",
            "page_number",
        ],
        how="left",
    )
    .to_csv(
        missing_ranks_path,
        index=False,
    )
)


print("Failure records:", len(failure_summary))
print("Missing-page records:", len(component_ranks))
print("Saved:", failures_path)
print("Saved:", missing_ranks_path)